# Day 24: Data Type Cleaning

## Description
Identify incorrect data types in a dataset and convert columns to appropriate data types.

## Objective
Understand why correct data types are crucial for efficient data analysis, memory optimization, and machine learning model training.

In [1]:
import pandas as pd

# Creating a messy sample Employee Dataset
data = {
    'Emp_ID': [101, 102, 103, 104, 105],
    'Name': ['Alice Smith', 'Bob Jones', 'Charlie Brown', 'Diana Prince', 'Evan Wright'],
    'Age': ['29', '34', 'not_available', '41', '28'],  # Contains a string anomaly
    'Salary': ['$65,000', '$72,000', '$54,500', '$89,000', '$62,000'], # String with symbols
    'Joining_Date': ['2021-06-15', '2019-11-20', '2022-01-10', '2018-05-30', '2023-03-12'], # Object/string type
    'Is_Active': ['True', 'False', 'True', 'True', 'False'] # Boolean stored as string
}

df = pd.DataFrame(data)
df.to_csv('employee_dataset.csv', index=False)
print("Sample dataset 'employee_dataset.csv' created and saved successfully!")
df.head()

Sample dataset 'employee_dataset.csv' created and saved successfully!


,Emp_ID,Name,Age,Salary,Joining_Date,Is_Active
0,101,Alice Smith,29,"$65,000",2021-06-15,True
1,102,Bob Jones,34,"$72,000",2019-11-20,False
2,103,Charlie Brown,not_available,"$54,500",2022-01-10,True
3,104,Diana Prince,41,"$89,000",2018-05-30,True
4,105,Evan Wright,28,"$62,000",2023-03-12,False


## Step 1: Loading and Inspecting Original Data Types
We will load our dataset and inspect the columns using `dtypes` and `info()` to identify which columns have incorrect data types.

In [2]:
# Load the dataset
df = pd.read_csv('employee_dataset.csv')

# Display original data types
print("--- ORIGINAL DATA TYPES ---")
print(df.dtypes)
print("\n--- DATASET INFO ---")
df.info()

--- ORIGINAL DATA TYPES ---
Emp_ID          int64
Name              str
Age               str
Salary            str
Joining_Date      str
Is_Active        bool
dtype: object

--- DATASET INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Emp_ID        5 non-null      int64
 1   Name          5 non-null      str  
 2   Age           5 non-null      str  
 3   Salary        5 non-null      str  
 4   Joining_Date  5 non-null      str  
 5   Is_Active     5 non-null      bool 
dtypes: bool(1), int64(1), str(4)
memory usage: 499.0 bytes


## Step 2: Correcting Data Types
Based on our inspection, we observe the following issues:
1. **`Age`**: Stored as `object` (string) and contains a non-numeric value (`'not_available'`). We will coerce errors to `NaN` and convert to `Int64` (nullable integer).
2. **`Salary`**: Stored as `object` because of `$` and `,`. We need to strip these characters and convert to `float` or `int`.
3. **`Joining_Date`**: Stored as `object`. We will convert it to `datetime64[ns]`.
4. **`Is_Active`**: Stored as `object`. We will convert it to boolean (`bool`).

In [4]:
# 1. Clean and convert 'Age' to numeric (handling bad strings safely)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# 2. Clean and convert 'Salary' to numeric by safely stripping '$' and ',' using regex=False
df['Salary'] = pd.to_numeric(
    df['Salary'].str.replace('$', '', regex=False).str.replace(',', '', regex=False), 
    errors='coerce'
)

# 3. Convert 'Joining_Date' to datetime
df['Joining_Date'] = pd.to_datetime(df['Joining_Date'])

# 4. Convert 'Is_Active' to boolean
df['Is_Active'] = df['Is_Active'].map({'True': True, 'False': False})

print("--- CORRECTED DATA TYPES ---")
print(df.dtypes)
print("\n")
df.info()

--- CORRECTED DATA TYPES ---
Emp_ID                   int64
Name                       str
Age                    float64
Salary                   int64
Joining_Date    datetime64[us]
Is_Active               object
dtype: object


<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Emp_ID        5 non-null      int64         
 1   Name          5 non-null      str           
 2   Age           4 non-null      float64       
 3   Salary        5 non-null      int64         
 4   Joining_Date  5 non-null      datetime64[us]
 5   Is_Active     0 non-null      object        
dtypes: datetime64[us](1), float64(1), int64(2), object(1), str(1)
memory usage: 428.0+ bytes


## Explanation of Conversions
* **`pd.to_numeric(..., errors='coerce')`**: This function safely converts columns with mixed or dirty text data into numbers, replacing invalid entries (like `'not_available'`) with `NaN` so calculations don't crash.
* **String Manipulation (`.str.replace()`)**: Removed currency symbols (`$`) and comma separators (`,`) from the `Salary` column so Pandas could recognize it as a continuous numerical feature.
* **`pd.to_datetime()`**: Standardizes date strings into datetime objects, enabling time-series analysis, calculating tenure, and sorting chronologically.
* **Mapping/Boolean Conversion**: Converted categorical text representations of true/false into proper boolean types, reducing memory usage and ensuring correct logical evaluations.

## Interview Questions & Answers

### Q1: What is a data type?
**Answer:** A data type is an attribute of data that tells the computer (and programming languages like Python/Pandas) how the programmer intends to interpret the data. Common data types include integers (`int`), floating-point numbers (`float`), strings/text (`object`), booleans (`bool`), and datetimes (`datetime64`).

### Q2: Why should numerical data have a numerical data type?
**Answer:** 
* **Mathematical Operations:** Numerical operations (like `sum()`, `mean()`, addition, or multiplication) can only be performed efficiently on numerical types (`int` or `float`). If numbers are stored as strings, operations will fail or produce concatenation errors (e.g., `'10' + '20'` results in `'1020'` instead of `30`).
* **Memory Optimization:** Numerical types take up predictable, optimized storage space compared to generic object/string arrays.
* **Machine Learning Compatibility:** Machine learning algorithms (such as Scikit-Learn models) strictly require numerical inputs to compute distances, gradients, and weights.

### Q3: How can you convert a column to numeric?
**Answer:** 
You can convert a column to numeric using:
1. **`pd.to_numeric(df['column_name'], errors='coerce')`**: The safest approach, especially for dirty real-world data, because `errors='coerce'` automatically turns unparsable values into `NaN` instead of raising an error.
2. **`df['column_name'].astype(float)`** or **`astype(int)`**: Best used when you are certain the column contains clean data without any invalid or missing text entries.